In [6]:
import plotly.express as px
import pandas as pd
import itertools


In [2]:
GloHydroRes = pd.read_excel("/home/shah0012/GloHydroRes/Output_data/GloHydroRes_vs2.xlsx", sheet_name="Data")

In [3]:
# For selected countries, extract the data from the GloHydroRes dataset
interest_countries = ['Germany', 'France', 'United Kingdom', 'Italy', 'Spain', 'Turkey']
selected_data = GloHydroRes[GloHydroRes['country'].isin(interest_countries)]

In [4]:
# For each country, sum the capacity of each year
selected_data = selected_data.groupby(['country', "year"])["capacity_mw"].sum().reset_index()

In [5]:
selected_data['year'] = selected_data.year.astype(int)

In [7]:
full_year_range = range(selected_data.year.min(), selected_data.year.max() + 1)

# Create a DataFrame with all combinations of countries and years
all_combinations = pd.DataFrame(list(itertools.product(interest_countries, full_year_range)), columns=['country', 'year'])

In [12]:
selected_data_updated  = selected_data.set_index(['country', 'year']).reindex(all_combinations, fill_value=0).reset_index()
selected_data_updated['cumulative_capacity_mw'] = selected_data_updated.groupby('country')["capacity_mw"].cumsum()
selected_data_updated["cumulative_capacity_gw"] = selected_data_updated.cumulative_capacity_mw*0.001

In [38]:
#animation_frame = This means year will be used to change the frames. So each year will be a frame.
#animation_group = Each country will represented in each frame.
fig = px.scatter(selected_data_updated, x="year", y="cumulative_capacity_gw", animation_frame="year", 
                 animation_group="country", size="cumulative_capacity_gw", color="country", 
                 hover_name="country", size_max=55, range_x=[selected_data_updated['year'].min(), selected_data_updated['year'].max()], 
                 range_y=[0, selected_data_updated['cumulative_capacity_gw'].max()], 
                 title="Cumulative Capacity Over Time",
                 labels={"cumulative_capacity_gw": "Cumulative Capacity (GW)", "year": "Year"})

fig.update_traces(showlegend=True, marker=dict(size=15))

fig.update_layout(legend_title_text='Country')
fig.show()
fig.write_html("/home/shah0012/GloHydroRes/Figure/animation.html")
